In [2]:
import pandas as pd
import sqlite3

In [3]:
connect = sqlite3.connect('../data/checking-logs.sqlite')
query = """SELECT * FROM test LIMIT 10"""

In [4]:
test = pd.io.sql.read_sql(query, connect)
test

,index,uid,labname,first_commit_ts,first_view_ts
0,3,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
1,4,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 12:05:48.200144
2,5,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 12:06:13.237290
3,6,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 12:06:35.574114
4,7,user_17,project1,2020-04-18 07:56:45.408648,2020-04-21 19:04:25.970852
5,8,user_17,project1,2020-04-18 07:56:45.408648,2020-04-26 18:11:53.804713
6,9,user_17,project1,2020-04-18 07:56:45.408648,2020-05-02 13:21:34.579038
7,10,user_17,project1,2020-04-18 07:56:45.408648,2020-05-11 10:55:04.335091
8,11,user_17,project1,2020-04-18 07:56:45.408648,2020-05-11 11:12:26.903732
9,12,user_17,project1,2020-04-18 07:56:45.408648,2020-05-12 19:31:22.618018


In [5]:
query = 'PRAGMA table_info(deadlines);'
pd.io.sql.read_sql(query, connect)

,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,labs,TEXT,0,None,0
2,2,deadlines,INTEGER,0,None,0


In [6]:
query = 'select * from test'
pd.io.sql.read_sql(query, connect)

,index,uid,labname,first_commit_ts,first_view_ts
0,3,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
1,4,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 12:05:48.200144
2,5,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 12:06:13.237290
3,6,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 12:06:35.574114
4,7,user_17,project1,2020-04-18 07:56:45.408648,2020-04-21 19:04:25.970852
...,...,...,...,...,...
5578,5659,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-15 13:01:22.753180
5579,5660,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-17 17:39:50.896297
5580,5661,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-18 20:47:36.971195
5581,5662,user_17,laba06s,2020-05-21 17:39:17.783615,2020-05-18 22:05:00.106464


Найдите минимальное значение разницы между первым коммитом и крайним сроком выполнения соответствующей лабораторной работы для всех пользователей, используя всего один запрос.

Для этого соедините одну таблицу с другой deadlines.
Разница должна отображаться в часах.
Не стоит project1учитывать лабораторные работы; у них более длительные сроки сдачи, и они будут выходить за рамки установленных норм.
Значение должно быть сохранено в датафрейме df_minвместе с соответствующим uid.

In [7]:
query = """
SELECT uid,
MIN(CAST((strftime('%s', deadlines.deadlines, 'unixepoch') - 
                 strftime('%s', test.first_commit_ts)) / 3600 AS INTEGER
               )) AS delta
FROM test
JOIN deadlines on test.labname = deadlines.labs
WHERE test.labname != 'project1'
"""
df_min = pd.io.sql.read_sql(query, connect)
df_min

,uid,delta
0,user_25,2


In [8]:
query = """
SELECT uid,
MAX(CAST((strftime('%s', deadlines.deadlines, 'unixepoch') -
            strftime('%s', test.first_commit_ts)) / 3600 AS INTEGER))
            AS delta
FROM test
JOIN deadlines on test.labname = deadlines.labs
WHERE test.labname != 'project1'
"""
df_max = pd.io.sql.read_sql(query, connect)
df_max

,uid,delta
0,user_30,202


In [9]:
query = """
SELECT
AVG(CAST((strftime('%s', deadlines.deadlines, 'unixepoch') -
            strftime('%s', test.first_commit_ts)) / 3600 AS INTEGER))
            AS delta
FROM test
JOIN deadlines on test.labname = deadlines.labs
WHERE test.labname != 'project1'
"""
df_avg = pd.io.sql.read_sql(query, connect)
df_avg

,delta
0,98.114155


In [10]:
query = '''
SELECT test.uid,
        AVG(CAST((strftime('%s', deadlines.deadlines, 'unixepoch') -
            strftime('%s', test.first_commit_ts)) / 3600 AS INTEGER)) AS delta,
            pageviews
FROM test
LEFT JOIN deadlines ON test.labname=deadlines.labs
LEFT JOIN (SELECT uid, count(*) AS pageviews
           FROM pageviews
           GROUP BY uid) AS views ON test.uid=views.uid
WHERE labname != 'project1'
GROUP BY test.uid
'''
views_diff = pd.io.sql.read_sql(query, connect)
views_diff

,uid,delta,pageviews
0,user_1,64.400000,28
1,user_10,74.800000,89
2,user_14,159.000000,143
3,user_17,61.600000,47
4,user_18,5.666667,3
5,user_19,98.750000,16
6,user_21,95.500000,10
7,user_25,92.600000,179
8,user_28,86.400000,149
9,user_3,105.400000,317


In [11]:
views_diff.corr()

ValueError: could not convert string to float: 'user_1'

In [ ]:
connect.close()